# Milestone 3 — Matching Pipeline Analysis & Model Evaluation

Notebook này do **Member 4 (Data Quality, Matching Model và Evaluation)** phụ trách nhằm:
1. **Feature Engineering Analysis:** Trực quan hóa và biện minh (justify) cho các đặc trưng tương đồng tiêu đề, tác giả, năm và định danh.
2. **Model Training & Comparison:** So sánh hiệu năng giữa **Baseline Heuristic Ranker** và các mô hình học máy **Learned Rankers (Logistic Regression, Random Forest)**.
3. **Metric MRR@5:** Đánh giá chất lượng xếp hạng Top-5 trên tập Validation và Test.
4. **Feature Importance & Ablation:** Cung cấp số liệu phân tích và biểu đồ cho **Member 5** đưa vào Poster đồ án.

## 1. Cài đặt thư viện và Import Modules

In [ ]:
!pip install -q pyarrow pandas scikit-learn matplotlib seaborn pytest

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import FEATURE_NAMES, extract_pair_features
from src.model import (
    HeuristicRanker,
    LearnedRanker,
    build_pairwise_dataset,
    predict_top_k_candidates,
    generate_mock_matching_dataset,
)
from src.evaluate import compute_mrr_at_k
from src.storage import create_table_from_records, write_sharded_parquet, PREDICTIONS_SCHEMA, BIBTEX_SCHEMA, REFERENCES_SCHEMA, PAPERS_SCHEMA
from src.validate import validate_predictions_table

sns.set_theme(style="whitegrid")
print("Đã import thành công toàn bộ module ML Matching Pipeline.")

## 2. Nạp dữ liệu hoặc Khởi tạo Sandbox Dataset
*(Nếu chưa có đủ dữ liệu crawl thật từ cả nhóm, sử dụng hàm `generate_mock_matching_dataset` để thử nghiệm pipeline ngay lập tức)*

In [ ]:
# Khởi tạo bộ dữ liệu giả lập có độ nhiễu thực tế (60% Exact, 25% Fuzzy/Typo, 15% No-match)
dataset = generate_mock_matching_dataset(num_pubs=20, refs_per_pub=15, entries_per_pub=15, seed=42)

papers = dataset["papers"]
references = dataset["references"]
bibtex = dataset["bibtex"]

print(f"Tổng số Publications: {len(papers)}")
print(f"Tổng số Crawled References: {len(references)}")
print(f"Tổng số BibTeX Entries: {len(bibtex)}")

## 3. Phân chia tập dữ liệu (Publication-Level Split)
*(Chia tập theo cấp publication: 10 bài Train, 5 bài Validation, 5 bài Test)*

In [ ]:
pub_ids = [p["arxiv_id"] for p in papers]
train_pubs = set(pub_ids[:10])
valid_pubs = set(pub_ids[10:15])
test_pubs = set(pub_ids[15:])

split_manifest = {}
for pid in train_pubs: split_manifest[pid] = "train"
for pid in valid_pubs: split_manifest[pid] = "valid"
for pid in test_pubs: split_manifest[pid] = "test"

train_entries = [e for e in bibtex if e["arxiv_id"] in train_pubs]
valid_entries = [e for e in bibtex if e["arxiv_id"] in valid_pubs]
test_entries = [e for e in bibtex if e["arxiv_id"] in test_pubs]

print(f"Train entries: {len(train_entries)}, Valid entries: {len(valid_entries)}, Test entries: {len(test_entries)}")

## 4. Phân tích Phân phối Đặc trưng (Feature Analysis & Justification)
*(Trực quan hóa sự khác biệt giữa các cặp Match (y=1) và Non-Match (y=0))* 

In [ ]:
# Trích xuất ma trận đặc trưng tập Train
X_train, y_train, meta_train = build_pairwise_dataset(train_entries, references, negative_sample_ratio=5, seed=42)
df_features = pd.DataFrame(X_train, columns=FEATURE_NAMES)
df_features["label"] = y_train

# Trực quan hóa phân phối một số đặc trưng then chốt
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df_features, x="label", y="title_token_jaccard", ax=axes[0, 0])
axes[0, 0].set_title("Title Token Jaccard by Match Status")

sns.boxplot(data=df_features, x="label", y="title_edit_ratio", ax=axes[0, 1])
axes[0, 1].set_title("Title Edit Ratio by Match Status")

sns.boxplot(data=df_features, x="label", y="author_set_overlap", ax=axes[1, 0])
axes[1, 0].set_title("Author Set Overlap by Match Status")

sns.boxplot(data=df_features, x="label", y="year_proximity", ax=axes[1, 1])
axes[1, 1].set_title("Year Proximity by Match Status")

plt.tight_layout()
plt.show()

## 5. Huấn luyện Mô hình & So sánh trên tập Validation

In [ ]:
# 1. Baseline: Heuristic Ranker
heuristic_ranker = HeuristicRanker()
preds_heur_val = predict_top_k_candidates(
    valid_entries, references, model=heuristic_ranker, split_manifest=split_manifest
)
mrr_heur = compute_mrr_at_k(preds_heur_val, k=5, partition_filter="valid")

# 2. Learned Model: Logistic Regression
lr_ranker = LearnedRanker(model_type="logistic", random_state=42)
lr_ranker.fit(X_train, y_train)
preds_lr_val = predict_top_k_candidates(
    valid_entries, references, model=lr_ranker, split_manifest=split_manifest
)
mrr_lr = compute_mrr_at_k(preds_lr_val, k=5, partition_filter="valid")

# 3. Learned Model: Random Forest
rf_ranker = LearnedRanker(model_type="random_forest", n_estimators=100, max_depth=6, random_state=42)
rf_ranker.fit(X_train, y_train)
preds_rf_val = predict_top_k_candidates(
    valid_entries, references, model=rf_ranker, split_manifest=split_manifest
)
mrr_rf = compute_mrr_at_k(preds_rf_val, k=5, partition_filter="valid")

# Bảng so sánh kết quả
comparison_df = pd.DataFrame([
    {"Model": "Baseline Heuristic", "MRR@5": mrr_heur["mrr"], "Hit Rate": mrr_heur["hit_rate"]},
    {"Model": "Logistic Regression", "MRR@5": mrr_lr["mrr"], "Hit Rate": mrr_lr["hit_rate"]},
    {"Model": "Random Forest", "MRR@5": mrr_rf["mrr"], "Hit Rate": mrr_rf["hit_rate"]},
])
display(comparison_df)

## 6. Phân tích Feature Importance (Cho Báo cáo & Poster)

In [ ]:
# Trích xuất Feature Importance từ Random Forest
importances = rf_ranker.model.feature_importances_
df_imp = pd.DataFrame({"Feature": FEATURE_NAMES, "Importance": importances})
df_imp = df_imp.sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_imp, x="Importance", y="Feature", palette="viridis")
plt.title("Random Forest Feature Importance in Reference Matching")
plt.tight_layout()
plt.show()

## 7. Đánh giá Final trên Tập Test, Lưu Sharded Parquet & Quality Gates

In [ ]:
# Chọn model tốt nhất (ví dụ Random Forest) để dự đoán trên toàn bộ các tập
best_model = rf_ranker if mrr_rf["mrr"] >= mrr_lr["mrr"] else lr_ranker
all_preds = predict_top_k_candidates(
    bibtex, references, model=best_model, split_manifest=split_manifest
)

# Đánh giá chính thức trên tập Test
test_mrr_res = compute_mrr_at_k(all_preds, k=5, partition_filter="test")
print("=== FINAL TEST MRR@5 RESULTS ===")
print(f"Total Test Queries: {test_mrr_res['total_queries']}")
print(f"MRR@5: {test_mrr_res['mrr']}")
print(f"Hit Rate: {test_mrr_res['hit_rate']}")
print(f"Rank Distribution: {test_mrr_res['rank_distribution']}")

# Kiểm tra Quality Gate cho bảng predictions
table_papers = create_table_from_records(papers, PAPERS_SCHEMA)
table_refs = create_table_from_records(references, REFERENCES_SCHEMA)
table_bib = create_table_from_records(bibtex, BIBTEX_SCHEMA)
table_preds = create_table_from_records(all_preds, PREDICTIONS_SCHEMA)

report = validate_predictions_table(
    table_preds,
    bibtex_table=table_bib,
    references_table=table_refs,
    papers_table=table_papers,
)
print(f"Predictions Table Validation: Valid={report.is_valid}, Errors={report.errors}")
assert report.is_valid, "Bảng predictions không vượt qua Quality Gate!"

# 1. Ghi bảng predictions ra Parquet phân mảnh chuẩn bị cho Hugging Face Dataset
data_dir = Path("/content/arxiv-data-benchmark") if os.path.exists("/content/arxiv-data-benchmark") else Path("./data-benchmark")
processed_dir = data_dir / "processed" if (data_dir / "processed").exists() else data_dir

pred_shards = write_sharded_parquet(table_preds, processed_dir, "predictions")
print(f"Đã lưu thành công {len(pred_shards)} shards cho bảng 'predictions' tại: {processed_dir / 'predictions'}")

# 2. Xuất gói dữ liệu thống kê M3 đồng thời ở cả code root và data dir cho Member 5
m3_summary = {
    "model_selected": "Random Forest" if best_model == rf_ranker else "Logistic Regression",
    "test_mrr": test_mrr_res,
    "validation_comparison": comparison_df.to_dict(orient="records"),
    "feature_importances": df_imp.to_dict(orient="records") if "df_imp" in locals() else {},
}
m3_summary_locations = [
    Path("metrics/matching_summary.json"),
    data_dir / "metrics" / "matching_summary.json",
]
for loc in m3_summary_locations:
    loc.parent.mkdir(parents=True, exist_ok=True)
    with open(loc, "w", encoding="utf-8") as f:
        json.dump(m3_summary, f, indent=2, ensure_ascii=False)
print(f"Đã xuất kết quả phân tích M3 cho Member 5 tại: {m3_summary_locations[0]}")